In [1]:
import pandas as pd
import numpy as np
from Bio import SeqIO, Seq
from Bio.SeqRecord import SeqRecord
from pyfaidx import Fasta
import pickle
import re
import random
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
from time import time
import os

In [2]:
# chrs = Fasta("/tamir2/shaicohen1/EXPosition/code/Data/Chromosome/hg38.fa")
# seq_records = list(SeqIO.parse("/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/features/shrimp_features/Prawn_relevant_scaffolds.fasta", "fasta"))
# shrimp_genome = pd.DataFrame([{"seq": str(s.seq), "id": s.id, "chromosome": s.name, "description": s.description} for s in seq_records])
# shrimp_genome["seq"] = shrimp_genome["seq"].str.upper()
# seq_records = list(SeqIO.parse('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/features/tomato_features/M82_MAS2.0.fasta', "fasta"))
# tom_genome = pd.DataFrame([{"seq": str(s.seq), "id": s.id, "chromosome": s.name, "description": s.description} for s in seq_records])
# tom_genome["seq"] = tom_genome["seq"].str.upper()
# tom_genome["chromosome"] = tom_genome["description"].apply(lambda x: re.findall("M82_MAS2.0ch(\d+)", x)).apply(lambda x: int(x[0]) if x else "chloroplast")
# print('done')

done


In [2]:
#DNA shape + enthalpy
with open("/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/DNAshape_enthalpy_test/dnaShape.pkl", "rb") as file:
    DNASHAPE_DICT = pickle.load(file)
DNA_PAIRS_THERMODYNAMICS = {"AA": 9.1, "AT": 8.6, "TA": 6.0, "CA": 5.8, "GT": 6.5, "CT": 7.8, "GA": 5.6, "CG": 11.9,
							"GC": 11.1, "GG": 11.0, "TT": 9.1, "TG": 5.8, "AC": 6.5, "AG": 7.8, "TC": 5.6, "CC": 11.0} #Breslauer et al.

In [3]:
def get_avg(l):
	return sum(l)/float(len(l))
def get_DNAshape_features(dna_seq):
    # """
    # :param dna_seq: sequence of nucleotides
    # :return: a dictionary with scores of rigidity for Major Groove Width (MGW), ProT (Propeller-Twist), Roll, and HelT (Helical-Twist).
    # The values are the scores for each pentamer/hexamer as computed by DNAshape (Zhou et al., doi:10.1093/nar/gkt437)
    # across the DNA sequence
    # """
    
    mgw = [None]
    roll = [None]
    prot = [None]
    helt = [None]

    for i in range(2, len(dna_seq) - 2):
        current_heptamer = dna_seq[i - 2: i + 3]
        current_heptamer = re.sub("N", random.choice(["A", "C", "G", "T"]), current_heptamer)
        current_nucleotide = DNASHAPE_DICT[current_heptamer]
        mgw += current_nucleotide["MGW"]
        roll += current_nucleotide["Roll"]
        prot += current_nucleotide["ProT"]
        helt += current_nucleotide["HelT"]

    ####ORIG#####
    # helt_modified = [helt[1]]
    helt_modified = []
    for i in range(2, len(helt), 2):
        helt_modified.append(get_avg(helt[i:i + 2]))
    # roll_modified = [roll[1]]
    roll_modified = []
    for i in range(2, len(roll), 2):
        roll_modified.append(get_avg(roll[i:i + 2]))
    return {"MGW": mgw[1:], "ProT": prot[1:], "Roll": roll_modified, "HelT": helt_modified}

In [9]:
tomato_genome = Fasta('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/cb_chimera_code/tomato_data/SollycM82_v1.0.fasta')
# shrimp_genome = Fasta('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/cb_chimera_code/shrimp_data/GCF_040412425.1_ASM4041242v1_genomic.fna')
# fly_genome = Fasta('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/cb_chimera_code/fly_data/GCF_905115235.1_iHerIll2.2.curated.20191125_genomic.fna')

In [4]:
#define columns:
def redefine_df(df):
    cols_to_drop = ['DNAshape','enthalpy']
    df = df.drop(columns=[col for col in df.columns if any(x in col for x in cols_to_drop)])
    for x in ['MGW', 'ProT', 'Roll', 'HelT']:
        columns_to_add = [f'DNAshape_{x}_{i}' for i in range(1,20)]+\
        [f'DNAshape_{x}_mean'] + [f'DNAshape_{x}_mean_ext'] + \
        [f'DNAshape_{x}_mean_up_ext'] + [f'DNAshape_{x}_mean_down_ext']
        new_cols_df = pd.DataFrame({col: np.nan for col in columns_to_add}, index=df.index)
        df = pd.concat([df, new_cols_df], axis=1)

    columns_to_add = [f'enthalpy_{i}' for i in range(1,23)]+\
    [f'enthalpy_mean'] + [f'enthalpy_mean_ext'] + \
    [f'enthalpy_mean_up_ext'] + [f'enthalpy_mean_down_ext']
    new_cols_df = pd.DataFrame({col: np.nan for col in columns_to_add}, index=df.index)
    df = pd.concat([df, new_cols_df], axis=1)
    return df


In [5]:
def gen_seq(chrm,start,end,strand,seq_type,organism='human'):
    if organism=='human':
        chrm_prefix = 'chr'
        genome = human_genome
    else:
        chrm_prefix = ''
    if organism=='fly':
        genome = fly_genome
    elif organism=='shrimp':
        genome = shrimp_genome
    elif organism=='tomato':
        genome = tomato_genome      
    if seq_type=='target':
        if strand=='+':
            start_delta = 0
            end_delta = 3
        else:
            start_delta = 3
            end_delta = 0
    elif seq_type=='extended':
        if strand=='+':
            start_delta = 100
            end_delta = 103
        else:
            start_delta = 103
            end_delta = 100
    elif seq_type=='down':
        if strand=='+':
            start_delta = 0
            end_delta = 103
        else:
            start_delta = 103
            end_delta = 0        
    elif seq_type=='up':
        if strand=='+':
            start_delta = 100
            end_delta = 0
        else:
            start_delta = 0
            end_delta = 100        
    seq = str(genome[chrm_prefix+chrm][start-start_delta:end+end_delta])
    if strand =='-':
        x = Seq.Seq(seq)
        x = x.reverse_complement()
        seq = str(x)
    seq = seq.upper()
    return seq

In [6]:
def no_N_seq(seq):
    num_N = sum([1 for x in seq if x=='N'])
    if num_N>3:
        seq = ''
    else:
        seq = re.sub("N", random.choice(["A", "C", "G", "T"]), seq)
    return seq

In [7]:
def gen_feats(chrm,start,end,strand, organism, method='all'):
    site_feats = redefine_df(pd.DataFrame({}))
    target_seq = no_N_seq(gen_seq(chrm,start,end,strand,'target',organism))
    extended_seq = no_N_seq(gen_seq(chrm,start,end,strand,'extended',organism))
    up_seq = no_N_seq(gen_seq(chrm,start,end,strand,'up',organism))
    down_seq = no_N_seq(gen_seq(chrm,start,end,strand,'down',organism))
    if method=='dnashape' or method=='all':
        if target_seq!='':
            target_DNAshape = get_DNAshape_features(target_seq)
        if extended_seq!='':            
            extended_DNAshape = get_DNAshape_features(extended_seq)
        if up_seq!='':
            up_DNAshape = get_DNAshape_features(up_seq)
        if down_seq!='':
            down_DNAshape = get_DNAshape_features(down_seq)
        for dna_attr in ['MGW', 'ProT', 'Roll', 'HelT']:
            if target_seq!='':
                site_feats.loc[0,f'DNAshape_{dna_attr}_mean'] = np.mean(target_DNAshape[dna_attr])
                site_feats.loc[0,[f'DNAshape_{dna_attr}_{i}' for i in range(1,20)]] = target_DNAshape[dna_attr]
            if extended_seq!='':            
                site_feats.loc[0,f'DNAshape_{dna_attr}_mean_ext'] = np.mean(extended_DNAshape[dna_attr])
            if up_seq!='':
                site_feats.loc[0,f'DNAshape_{dna_attr}_mean_up_ext'] = np.mean(up_DNAshape[dna_attr])
            if down_seq!='':
                site_feats.loc[0,f'DNAshape_{dna_attr}_mean_down_ext'] = np.mean(down_DNAshape[dna_attr])
    if method=='enthalpy' or method=='all':
        if target_seq!='': 
            site_feats.loc[0,[f'enthalpy_{i}' for i in range(1,23)]] = [DNA_PAIRS_THERMODYNAMICS[target_seq[i-1:i+1]] for i in range(1, len(target_seq))]
            site_feats[f'enthalpy_mean'] = np.mean([DNA_PAIRS_THERMODYNAMICS[target_seq[i-1:i+1]] for i in range(1, len(target_seq))])
        if extended_seq!='':
            site_feats[f'enthalpy_mean_ext'] = np.mean([DNA_PAIRS_THERMODYNAMICS[extended_seq[i-1:i+1]] for i in range(1, len(extended_seq))])
        if up_seq!='':
            site_feats[f'enthalpy_mean_up_ext'] = np.mean([DNA_PAIRS_THERMODYNAMICS[up_seq[i-1:i+1]] for i in range(1, len(up_seq))])
        if down_seq!='':
            site_feats[f'enthalpy_mean_down_ext'] = np.mean([DNA_PAIRS_THERMODYNAMICS[down_seq[i-1:i+1]] for i in range(1, len(down_seq))])
    return site_feats

In [10]:
path_data = '/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/DNAshape_enthalpy_test/new_shrimp_fly_tomato/'
list_files = os.listdir(path_data)
list_files = [x for x in list_files if '.csv' in x]

tomato = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/Isana_code/new_shrimp_fly_tomato/tomato_roots_w_isana.csv')
tomato=tomato.iloc[:10]
filename = 'tomato_hairy_roots'
organism='tomato'

for df in [tomato]:
# for filename,organism in zip(list_files,['fly','shrimp','tomato']):
#     t=time()
#     print(f'starting {filename}')
#     df = pd.read_csv(os.path.join(path_data, filename))
    df = redefine_df(df)
    for i in df.index:
        if i%1000==0:
            print(f'{i/df.shape[0]:.3f}')
        chrm,start,end,strand = df.loc[i,'g_rna_info'].split(';')
        site_df = gen_feats(chrm,int(start),int(end),strand,organism,'all')
        df.loc[i,site_df.columns] = site_df.loc[0,site_df.columns]
    # df = df.drop(columns=[x for x in df.columns if 'Unnamed' in x])
    # df.to_csv(path_data + f'{filename[:-4]}_w_DNAshape_enthalpy.csv', index=False)
    print('done')
    # print(time()-t)

0.000
done
